# Lab 6: Catalog the PlanetScope and SkySat imagery for the Iran war window

**Course:** Politics of World Energy Rivalry (Stanford, Policy)

**What you will do**

1. Load the six refinery site boundaries (areas of interest, or AOIs) from the `AOI/` folder.
2. Use the **Planet Python SDK** to search Planet's catalog for PlanetScope and SkySat imagery over each AOI.
3. Cover the period from **six months before the 2026 Iran war** (it began 28 February 2026) up to today.
4. Filter by **AOI coverage** and **cloud cover**.
5. Save a catalog (a table of every scene found) and see how much imagery exists, site by site and month by month.

**This notebook only searches. It does not order or download any imagery.** Searching is free and does not use your area quota. Lab 1 shows how to order clips once you know what exists.

**What you need**

- A Planet account with an API key (Stanford students: ask the Stanford Geospatial Center).
- Either Google Colab (the key goes in a Colab *secret*) or a local Python 3.10+ setup (the key goes in an *environment variable*). Step 3 walks you through both.
- SkySat search needs a Planet plan that includes SkySat. If you do not have it, that part returns zero scenes. The notebook tells you if that happens.

**Why this matters for policy:** Before you can say anything about a refinery's tank levels during the war, you need to know whether a satellite actually looked. This catalog shows which sites, and which months, have usable images. It also shows the gaps.

## Concepts you need first

**SDK (software development kit).** A package of ready-made code for working with a service. Planet's SDK is the Python package `planet`. Without it you would write raw web requests by hand (Lab 1 does this with `requests`). With it, you write one line such as `pl.data.search(...)` and the SDK handles the web details, the login, and the paging through long result lists.

**Data API.** The part of Planet's service that answers questions like "which scenes exist over this place and time?" The SDK reaches it through `pl.data`.

**Item type.** The kind of imagery. Each search names one or more item types.

| Product | Item type | Resolution | What it is |
|---|---|---|---|
| PlanetScope | `PSScene` | about 3 m per pixel | Hundreds of small satellites. Photographs most land about once a day. One record per scene. |
| SkySat | `SkySatCollect` | about 0.5 m per pixel | A few larger satellites that are *tasked* (pointed at a target on request). One record per **collect**, a strip of many scenes taken in one pass. |

SkySat is sharper but rarer. Expect far fewer SkySat records than PlanetScope records, and some sites may have none.

**Catalog search vs. order.** A search returns a list of scenes and their details (metadata). An order creates files to download. This notebook does only the search.

**AOI (area of interest).** A polygon that marks where you want data. Each AOI here is a 5 km by 5 km square centered on a refinery, stored as a GeoJSON file.

**Filters.** Rules that a scene must pass to be kept. In this lab:

| Filter | Question it answers | Setting |
|---|---|---|
| Geometry | Does the scene overlap the AOI at all? | always on |
| Date range | Was it taken inside the war window? | `WAR_START_DATE`, `MONTHS_BEFORE_WAR`, `END_DATE` |
| **Cloud cover** | How much of the scene is cloud? (0 = clear, 1 = fully cloudy) | `ENABLE_CLOUD_FILTER`, `MAX_CLOUD` |
| **Coverage** | What share of *our AOI* does the scene's footprint cover? | `ENABLE_COVERAGE_FILTER`, `MIN_COVERAGE` |

**Why coverage matters.** A scene "overlaps" the AOI even if it only clips one corner. A scene that covers 30 percent of the AOI leaves most of the refinery out of the picture. The coverage filter drops those.

**Why cloud cover is only an estimate.** The `cloud_cover` number describes the *whole scene* (about 600 km² for PlanetScope), not just your 25 km² AOI. A scene marked 40 percent cloudy can still be clear over your refinery, and a 5 percent scene can have a cloud sitting right on it. Treat the filter as a first screen, not a guarantee.

**API key.** A password that tells Planet who you are. Never paste it into a notebook you share, and never save it in a file that goes on GitHub.

## Step 1: Settings

**Every input for this notebook is in the cell below.** Change a value here, then rerun the cells below it. Nothing further down needs editing.

Run this cell first. It only stores values. It does not do anything yet.

In [ ]:
# ============================ INPUTS: edit these ============================

# --- Where files live -------------------------------------------------------
# These are paths relative to this notebook's folder (notebooks/).
# Running in Google Colab? Upload the AOI folder first (see Step 4), then use:
#     AOI_DIR = "/content/AOI"      OUT_DIR = "/content/planet_catalog"
AOI_DIR = "../AOI"                              # folder with the *_aoi.geojson files
OUT_DIR = "../data/imagery/planet_catalog"      # where the catalog CSV files are saved

# --- Time window ------------------------------------------------------------
# The window runs from MONTHS_BEFORE_WAR months before WAR_START_DATE
# up to END_DATE. Dates are written as "YYYY-MM-DD".
WAR_START_DATE = "2026-02-28"   # first day of the 2026 Iran war (US and Israeli strikes began)
MONTHS_BEFORE_WAR = 6           # how far back before the war to start the search
END_DATE = "today"              # "today", or a fixed date such as "2026-10-01"

# --- Which imagery to search ------------------------------------------------
# Label -> Planet "item type". Delete a line to skip that product.
PRODUCTS = {
    "PlanetScope": "PSScene",        # 3 m, near-daily
    "SkySat": "SkySatCollect",       # 0.5 m, tasked. Needs SkySat access on your plan.
}

# --- Filters ----------------------------------------------------------------
# Cloud cover filter: keep scenes whose cloud_cover is at or below MAX_CLOUD.
ENABLE_CLOUD_FILTER = True
MAX_CLOUD = 0.2                  # 0.2 = 20 percent cloud, on a 0 to 1 scale

# Coverage filter: keep scenes that cover at least MIN_COVERAGE of the AOI.
ENABLE_COVERAGE_FILTER = True
MIN_COVERAGE = 0.95              # 0.95 = the scene must cover 95 percent of the AOI

# Only list scenes your account is allowed to download?
# True  = what you could actually order. False = everything that exists in the catalog.
ONLY_DOWNLOADABLE = True

# --- Authentication (see Step 3) --------------------------------------------
# "colab" = read the key from a Google Colab secret named PL_API_KEY
# "env"   = read the key from an environment variable named PL_API_KEY
AUTH_METHOD = "env"

# ============================================================================

## Step 2: Install and import

The `%pip install` line downloads the Planet SDK (the `planet` package) into this notebook's Python. It is safe to run more than once. In Colab, `pandas`, `matplotlib`, `shapely` and `geopandas` are already installed. On your own computer, install them with `pip install -r requirements.txt`.

Each `import` line brings in a tool we use below.

In [ ]:
%pip install -q planet

In [ ]:
import os                        # read environment variables (where the API key can live)
import json                      # read the GeoJSON AOI files
from datetime import datetime, timezone   # build date values for the search
from pathlib import Path         # work with file paths on any operating system

import pandas as pd              # tables of data (DataFrames)
import matplotlib.pyplot as plt  # charts
import geopandas as gpd          # tables of map data (used to measure AOI area)
from shapely.geometry import shape   # turn GeoJSON shapes into Python objects

import planet                                   # the Planet SDK
from planet import Planet, Session, Auth, data_filter
#   Planet       - the main client. Its .data part searches the catalog.
#   Session      - holds the connection to Planet.
#   Auth         - holds your API key.
#   data_filter  - builders for search filters (geometry, dates, cloud, ...).

PL_API_KEY = None    # Step 3 fills this in

## Step 3: Log in to Planet

The SDK needs your **API key**. Planet gives each account one. Treat it like a password.

**Find your key:** sign in at [planet.com](https://www.planet.com), open your **Account settings**, and look for **API key**. (Stanford students without an account or key: ask the Stanford Geospatial Center.) Copy the long text that starts with `PLAK`.

There are two ways to hand the key to this notebook. **Use only the one that matches where you are working**, then set `AUTH_METHOD` in Step 1 to match:

| You are working in... | Use | Set `AUTH_METHOD` to |
|---|---|---|
| Google Colab | **Method A:** Colab secret | `"colab"` |
| Jupyter or VS Code on your own computer | **Method B:** environment variable | `"env"` |

Both methods do the same job: they keep the key *out of the notebook*. If your key were typed into a cell, it would be saved in the file, and anyone you share the notebook with could use your account and your quota.

You can run both auth cells. The one you did not choose just says it was skipped.

### Method A: Google Colab secret

A **Colab secret** is a private note stored in your Google account. Colab hands it to your notebook when asked, and it is never saved inside the notebook file.

**Create the secret (one time):**

1. Open this notebook in Colab.
2. In the **left sidebar**, click the **key icon** (labeled **Secrets**).
3. Click **+ Add new secret**.
4. In the **Name** box, type exactly `PL_API_KEY`. Capital letters and the underscore matter.
5. In the **Value** box, paste your Planet API key.
6. Switch **Notebook access** to **on** (the toggle next to the secret). If it is off, the notebook cannot see the key. This is the most common mistake.
7. In Step 1, set `AUTH_METHOD = "colab"`, then run the cell below. Colab may pop up a window asking you to grant access. Click **Grant access**.

**How the code works:** `userdata.get("PL_API_KEY")` asks Colab for the secret with that name and returns its value as text.

In [ ]:
# METHOD A: read the key from a Google Colab secret.
if AUTH_METHOD == "colab":
    try:
        from google.colab import userdata      # only exists inside Colab
    except ImportError:
        raise RuntimeError(
            "This notebook is not running in Google Colab, so Colab secrets are not available.\n"
            "Set AUTH_METHOD = 'env' in Step 1 and follow Method B instead."
        )

    try:
        PL_API_KEY = userdata.get("PL_API_KEY")   # ask Colab for the secret
    except Exception as err:
        # Colab raises an error if the secret is missing or access is off.
        raise RuntimeError(
            "Could not read the Colab secret named PL_API_KEY. Check that:\n"
            "  1. you clicked the key icon in the left sidebar and added a secret,\n"
            "  2. its name is exactly PL_API_KEY,\n"
            "  3. the 'Notebook access' toggle next to it is ON.\n"
            f"Original error: {err}"
        )
    print("Method A: key loaded from the Colab secret.")
else:
    print("Method A skipped (AUTH_METHOD is not 'colab').")

### Method B: environment variable named `PL_API_KEY`

An **environment variable** is a named value that your operating system keeps and hands to every program you start. Python reads it with `os.environ.get("PL_API_KEY")`. The key lives in your computer's settings, not in the notebook. The Planet SDK also looks for a variable with exactly this name on its own, which is why we use it.

**Create the variable (one time).** Pick your system:

**Mac (the default shell is zsh)**

1. Open the **Terminal** app.
2. Paste this line, with your real key in place of the placeholder, and press Return:
   ```
   echo 'export PL_API_KEY="paste-your-key-here"' >> ~/.zshrc
   ```
   This adds one line to a hidden settings file, `~/.zshrc`, that Terminal reads every time it opens.
3. Load it into the current window:
   ```
   source ~/.zshrc
   ```
4. Check that it worked. This should print your key:
   ```
   echo $PL_API_KEY
   ```

**Linux:** the same steps, but use `~/.bashrc` instead of `~/.zshrc`.

**Windows (PowerShell)**

1. Open **PowerShell** and run, with your real key:
   ```
   setx PL_API_KEY "paste-your-key-here"
   ```
2. Close PowerShell. `setx` does not change windows that are already open.
3. Open a new PowerShell and check it. This should print your key:
   ```
   echo $env:PL_API_KEY
   ```
   (No command line? Press the Windows key, type **Edit environment variables for your account**, click **New**, enter the name `PL_API_KEY` and your key as the value.)

**Then restart your notebook program.** This step is the one people forget. Programs only read environment variables *when they start*. Fully quit and reopen Jupyter or VS Code (on a Mac, quit with Cmd+Q, not just close the window), then reopen this notebook. If you start Jupyter from a terminal, start it from a terminal where `echo $PL_API_KEY` prints your key.

**Quick one-session alternative (Mac/Linux):** in a terminal, run `export PL_API_KEY="paste-your-key-here"`, then start Jupyter *from that same terminal window*. The variable disappears when you close it.

**Keep it private:** the key is saved as plain text in `~/.zshrc`. That is fine on your own laptop. Never copy that file into a GitHub repo or share it.

**If the cell below says the variable is not set, even though `echo $PL_API_KEY` works in Terminal:** the notebook's Python (the *kernel*) was started somewhere that never read your settings file. Mac apps launched from the Dock, Spotlight or Anaconda Navigator do not read `~/.zshrc`. Only programs started *from a Terminal window* do. Fix it like this:

1. In Terminal, activate your environment and confirm the key prints:
   ```
   conda activate geo_env
   echo $PL_API_KEY
   ```
2. Fully quit Jupyter or VS Code. Then start it **from that same Terminal window**:
   - JupyterLab: `jupyter lab`
   - VS Code: `code .` (run it in the project folder)
3. Reopen this notebook and choose **Restart** on the kernel (the restart button in the toolbar). A kernel reads the environment only when it starts, so rerunning cells is not enough.
4. Run the check cell below. It should print `True`.

Running `echo '...' >> ~/.zshrc` more than once adds the line again each time. That is harmless, but you can open `~/.zshrc` in a text editor and delete the duplicates.

**Check what the kernel can see:** this tells you whether the problem is the key or the launch method. It prints `True` or `False` and never shows the key.

In Step 1, set `AUTH_METHOD = "env"`, then run the check cell and the loading cell below.

In [ ]:
# Does THIS notebook's Python see the variable? (prints True or False, never the key)
print("PL_API_KEY visible to this kernel:", "PL_API_KEY" in os.environ)

In [ ]:
# METHOD B: read the key from an environment variable.
if AUTH_METHOD == "env":
    # os.environ.get returns the value, or "" if the variable does not exist.
    # .strip() removes stray spaces or line breaks picked up when pasting.
    PL_API_KEY = os.environ.get("PL_API_KEY", "").strip()

    if not PL_API_KEY:
        raise RuntimeError(
            "The environment variable PL_API_KEY is not set (or is empty).\n"
            "Follow the Method B steps above, then FULLY RESTART Jupyter or VS Code.\n"
            "Running in Colab? Set AUTH_METHOD = 'colab' in Step 1 and use Method A."
        )
    print("Method B: key loaded from the environment variable.")
else:
    print("Method B skipped (AUTH_METHOD is not 'env').")

### Connect and test the key

Now we hand the key to the SDK.

- `Auth.from_key(...)` wraps your key.
- `Session(auth=...)` opens a connection that sends the key with every request.
- `Planet(session=...)` is the client. Its `.data` part is the catalog search.

To test, we ask for a single PlanetScope scene anywhere in the world. If the key is wrong, Planet says so right away. We print only the first four characters of the key, never the whole thing.

In [ ]:
if AUTH_METHOD not in ("colab", "env"):
    raise ValueError("AUTH_METHOD must be 'colab' or 'env'. Check Step 1.")
if not PL_API_KEY:
    raise RuntimeError("No API key loaded. Run the auth cell for your method (A or B) above.")

PL_API_KEY = PL_API_KEY.strip()
print(f"Key loaded: starts with {PL_API_KEY[:4]}..., {len(PL_API_KEY)} characters")

# Build the client.
session = Session(auth=Auth.from_key(PL_API_KEY))
pl = Planet(session=session)

# Test call: fetch at most one scene. limit=1 means "stop after one result".
try:
    first = next(iter(pl.data.search(["PSScene"], limit=1)), None)
    print("Connected to Planet. The key works.")
except planet.exceptions.PlanetError as err:
    raise RuntimeError(
        "Planet rejected the request. The most common cause is a mistyped or "
        "expired API key. Check the key and try again.\n"
        f"Original error: {err}"
    )

## Step 4: Load the AOIs

Each `*_aoi.geojson` file holds one square. We read each into a Python dictionary keyed by a short site name (the "slug").

**In Colab:** the repo's files are not there automatically. Click the **folder icon** in the left sidebar, then drag the `AOI` folder from your computer into the file panel. Files you upload last only until the Colab session ends. Then set `AOI_DIR = "/content/AOI"` in Step 1.

In [ ]:
aoi_dir = Path(AOI_DIR).resolve()
out_dir = Path(OUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)     # create the output folder if missing

aois = {}   # will map site slug -> information about that site

for path in sorted(aoi_dir.glob("*_aoi.geojson")):
    data = json.loads(path.read_text())          # read the file as JSON
    feature = data["features"][0]                # each file has one feature
    slug = path.stem.replace("_aoi", "")         # e.g. "shell_pernis_netherlands"

    # Measure the area in km2. Degrees are not a unit of area, so we convert
    # the shape to a metric projection (UTM) first.
    gdf = gpd.GeoDataFrame(geometry=[shape(feature["geometry"])], crs="EPSG:4326")
    area_km2 = gdf.to_crs(gdf.estimate_utm_crs()).area.iloc[0] / 1e6

    aois[slug] = {
        "name": feature["properties"]["facility"],
        "lat": feature["properties"]["center_lat"],
        "lon": feature["properties"]["center_lon"],
        "geometry": feature["geometry"],          # GeoJSON dict, sent to Planet
        "shape": shape(feature["geometry"]),      # shapely object, used for math
        "area_km2": round(area_km2, 1),
    }

if not aois:
    raise FileNotFoundError(f"No *_aoi.geojson files found in {aoi_dir}. Check AOI_DIR in Step 1.")

pd.DataFrame(aois).T[["name", "lat", "lon", "area_km2"]]

## Step 5: Work out the search window

The settings give a war start date and a number of months. Here we turn them into a start and end date. `pd.DateOffset(months=6)` subtracts six calendar months, so 28 February 2026 minus six months is 28 August 2025.

In [ ]:
war_start = pd.Timestamp(WAR_START_DATE, tz="UTC")
window_start = war_start - pd.DateOffset(months=MONTHS_BEFORE_WAR)

if str(END_DATE).lower() == "today":
    window_end = pd.Timestamp.now(tz="UTC").normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
else:
    window_end = pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(hours=23, minutes=59, seconds=59)

if window_end <= window_start:
    raise ValueError("END_DATE must be after the start of the window. Check Step 1.")

print(f"War starts:     {war_start.date()}")
print(f"Search starts:  {window_start.date()}  ({MONTHS_BEFORE_WAR} months before the war)")
print(f"Search ends:    {window_end.date()}")
print(f"Window length:  {(window_end - window_start).days} days")

## Step 6: Search the catalog with the SDK

A search request is a set of filters joined by AND. A scene must pass every one to be returned. We build them with `data_filter`:

- `data_filter.geometry_filter(...)`: the scene must overlap the AOI.
- `data_filter.date_range_filter("acquired", gte=..., lte=...)`: taken on or after the start, on or before the end. (`gte` means "greater than or equal", `lte` means "less than or equal".)
- `data_filter.permission_filter()`: your account is allowed to download it (only if `ONLY_DOWNLOADABLE = True`).
- `data_filter.and_filter([...])`: joins them together.

Then `pl.data.search(["PSScene"], search_filter=..., limit=0)` runs the search. Two things to know:

- **The SDK handles paging for you.** Planet returns long results in pages of a few hundred. The SDK follows the pages and hands you one item at a time.
- **`limit=0` means "no limit".** The default limit is 100 scenes, which would silently cut a long search short.

**Why we apply cloud and coverage *after* the search.** We could ask Planet to drop cloudy scenes for us. Instead we fetch every scene in the window and measure cloud cover and coverage ourselves. That way the catalog also shows how much each filter removes, and a scene with a missing cloud value does not vanish without a trace. Step 7 applies the filters.

Coverage is calculated as: *area of (scene footprint ∩ AOI) ÷ area of AOI*. The two shapes are in degrees, which is fine for a ratio over an area this small.

In [ ]:
def build_filter(geometry, start, end):
    """Build the SDK search filter: AOI overlap + date range (+ permission)."""
    filters = [
        data_filter.geometry_filter(geometry),
        data_filter.date_range_filter(
            "acquired",
            gte=start.to_pydatetime(),    # the SDK wants Python datetime objects
            lte=end.to_pydatetime(),
        ),
    ]
    if ONLY_DOWNLOADABLE:
        filters.append(data_filter.permission_filter())
    return data_filter.and_filter(filters)


def items_to_table(items, aoi_shape, label, item_type, slug, site_name):
    """Turn raw search results (a list of dicts) into one table row per scene."""
    rows = []
    for item in items:
        p = item["properties"]                       # the scene's metadata
        footprint = shape(item["geometry"])          # the scene's outline
        if not footprint.is_valid:
            footprint = footprint.buffer(0)          # repair a malformed outline
        # Share of the AOI that this scene covers (0 to 1).
        coverage = footprint.intersection(aoi_shape).area / aoi_shape.area
        rows.append({
            "site": site_name,
            "slug": slug,
            "product": label,
            "item_type": item_type,
            "id": item["id"],
            "acquired": p.get("acquired"),
            "cloud_cover": p.get("cloud_cover"),
            "clear_percent": p.get("clear_percent"),
            "coverage": round(coverage, 3),
            "sun_elevation": p.get("sun_elevation"),
            "view_angle": p.get("view_angle"),
            "gsd_m": p.get("gsd") or p.get("pixel_resolution"),
            "satellite_id": p.get("satellite_id"),
            "quality_category": p.get("quality_category"),
        })
    table = pd.DataFrame(rows)
    if not table.empty:
        table["acquired"] = pd.to_datetime(table["acquired"], utc=True)   # text -> dates
        table = table.sort_values("acquired").reset_index(drop=True)
    return table

In [ ]:
tables = []      # one table per (site, product)
problems = []    # searches that failed, reported at the end

for label, item_type in PRODUCTS.items():
    print(f"\n=== {label} ({item_type}) ===")
    for slug, aoi in aois.items():
        search_filter = build_filter(aoi["geometry"], window_start, window_end)
        try:
            # list(...) pulls every item out of the SDK's one-at-a-time stream.
            items = list(pl.data.search([item_type], search_filter=search_filter, limit=0))
        except planet.exceptions.PlanetError as err:
            problems.append((label, aoi["name"], str(err)))
            print(f"  {aoi['name']:<35} SEARCH FAILED: {err}")
            continue
        table = items_to_table(items, aoi["shape"], label, item_type, slug, aoi["name"])
        tables.append(table)
        print(f"  {aoi['name']:<35} {len(table):>5} scenes found")

if problems:
    print("\nSome searches failed. A 403 or 'permission' message for SkySat usually means "
          "your plan does not include SkySat.")

nonempty = [t for t in tables if not t.empty]
catalog = pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame()
print(f"\nTotal scenes in catalog before filtering: {len(catalog)}")

## Step 7: Apply the cloud and coverage filters

We add true/false columns rather than deleting rows, so you can always see what was removed:

- `before_war`: taken before the war start date.
- `pass_cloud`: cloud cover is at or below `MAX_CLOUD` (always true if the filter is turned off).
- `pass_coverage`: coverage is at or above `MIN_COVERAGE` (always true if the filter is turned off).
- `pass_all`: passes every enabled filter.

**A scale check.** Planet reports `cloud_cover` on a 0 to 1 scale for PlanetScope. If a product ever reports 0 to 100 instead, the code below spots it (any value above 1) and divides by 100, and tells you.

In [ ]:
if catalog.empty:
    print("The catalog is empty. Nothing to filter. Check the dates, AOIs and your Planet access.")
else:
    # Scale check, one product at a time.
    for label in catalog["product"].unique():
        mask = catalog["product"] == label
        top = catalog.loc[mask, "cloud_cover"].max()
        if pd.notna(top) and top > 1:
            catalog.loc[mask, "cloud_cover"] = catalog.loc[mask, "cloud_cover"] / 100
            print(f"{label}: cloud_cover was on a 0-100 scale. Divided by 100.")

    missing = catalog["cloud_cover"].isna().sum()
    if missing:
        print(f"Note: {missing} scenes have no cloud_cover value. "
              f"They fail the cloud filter when it is enabled.")

    catalog["before_war"] = catalog["acquired"] < war_start

    if ENABLE_CLOUD_FILTER:
        catalog["pass_cloud"] = catalog["cloud_cover"] <= MAX_CLOUD
    else:
        catalog["pass_cloud"] = True

    if ENABLE_COVERAGE_FILTER:
        catalog["pass_coverage"] = catalog["coverage"] >= MIN_COVERAGE
    else:
        catalog["pass_coverage"] = True

    catalog["pass_all"] = catalog["pass_cloud"] & catalog["pass_coverage"]

    print(f"Cloud filter:    {'ON, max ' + str(MAX_CLOUD) if ENABLE_CLOUD_FILTER else 'OFF'}")
    print(f"Coverage filter: {'ON, min ' + str(MIN_COVERAGE) if ENABLE_COVERAGE_FILTER else 'OFF'}")
    print(f"{int(catalog['pass_all'].sum())} of {len(catalog)} scenes pass all enabled filters.")

## Step 8: What imagery exists?

**Table 1** counts scenes for each site and product: everything found, how many fall before and after the war began, and how many pass the filters.

In [ ]:
def summarize(df):
    """One summary row for a group of scenes."""
    ok = df[df["pass_all"]]
    return pd.Series({
        "found": len(df),
        "pass_cloud": int(df["pass_cloud"].sum()),
        "pass_coverage": int(df["pass_coverage"].sum()),
        "pass_all": len(ok),
        "usable_before_war": int(ok["before_war"].sum()),
        "usable_after_war": int((~ok["before_war"]).sum()),
        "first_usable": ok["acquired"].min().date() if len(ok) else None,
        "last_usable": ok["acquired"].max().date() if len(ok) else None,
    })

if not catalog.empty:
    rows = []
    for (product, site), group in catalog.groupby(["product", "site"]):
        row = summarize(group)
        row["product"], row["site"] = product, site
        rows.append(row)
    summary = pd.DataFrame(rows).set_index(["product", "site"]).reset_index()
    display(summary)

**Table 2** lists every site and product that has **no usable scene** after filtering, including sites with no scenes at all. These are your gaps.

In [ ]:
if not catalog.empty:
    # Build the full list of site x product pairs, then see which have zero usable scenes.
    wanted = pd.MultiIndex.from_product(
        [list(PRODUCTS), [a["name"] for a in aois.values()]], names=["product", "site"])
    usable = (catalog[catalog["pass_all"]].groupby(["product", "site"]).size()
              .reindex(wanted, fill_value=0))
    gaps = usable[usable == 0].reset_index()[["product", "site"]]
    print("Site and product pairs with no usable scenes:" if len(gaps) else "Every site has at least one usable scene for every product.")
    display(gaps)

## Step 9: Charts

**Chart 1** is a heat map of usable scenes per month. Rows are sites, columns are months, and each cell is a count of scenes that passed the filters. A dark red line marks the first day of the war, placed partway across its month. A cell showing 0 means no usable imagery that month, which is a gap in what you could say about the site.

Both products get their own panel. Notice the different scales: PlanetScope counts run to dozens per month, SkySat counts to a handful, if any.

In [ ]:
if catalog.empty:
    print("Nothing to chart.")
else:
    months = pd.period_range(window_start.tz_localize(None), window_end.tz_localize(None), freq="M")
    war_month = pd.Period(war_start.tz_localize(None), freq="M")
    site_names = [a["name"] for a in aois.values()]

    fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(13, 3.6 * len(PRODUCTS)), squeeze=False)
    for ax, label in zip(axes[:, 0], PRODUCTS):
        ok = catalog[(catalog["product"] == label) & catalog["pass_all"]].copy()
        ok["month"] = ok["acquired"].dt.tz_localize(None).dt.to_period("M")
        counts = (ok.groupby(["site", "month"]).size().unstack(fill_value=0)
                    .reindex(index=site_names, columns=months, fill_value=0))

        im = ax.imshow(counts.values, aspect="auto", cmap="Blues")
        ax.set_yticks(range(len(site_names)))
        ax.set_yticklabels(site_names, fontsize=9)
        ax.set_xticks(range(len(months)))
        ax.set_xticklabels([str(m) for m in months], rotation=60, fontsize=8)
        # Write the count inside each cell.
        for i in range(counts.shape[0]):
            for j in range(counts.shape[1]):
                n = counts.values[i, j]
                dark = n > counts.values.max() * 0.6
                ax.text(j, i, n, ha="center", va="center", fontsize=7,
                        color="white" if dark else "black")
        # Mark the war start. Each month is one column, so place the line the right
        # fraction of the way across the war month's column (28 Feb is near its right edge).
        if war_month in months:
            frac = (war_start.day - 1) / war_month.days_in_month
            ax.axvline(list(months).index(war_month) - 0.5 + frac, color="darkred", lw=2)
        ax.set_title(f"{label}: usable scenes per month (red line = war starts)", fontsize=11)
        fig.colorbar(im, ax=ax, label="scenes")
    plt.tight_layout()
    plt.show()

**Chart 2** shows every *usable* scene as a dot, with cloud cover on the vertical axis. Where dots are thick, you have near-daily imagery. Where there is a long empty stretch, you have a gap. Each panel is one site.

In [ ]:
if catalog.empty:
    print("Nothing to chart.")
else:
    colors = {"PlanetScope": "tab:blue", "SkySat": "tab:orange"}
    n = len(aois)
    cols = 3
    rows_n = -(-n // cols)     # round up
    fig, axes = plt.subplots(rows_n, cols, figsize=(14, 3.2 * rows_n), sharex=True, sharey=True,
                             squeeze=False)
    for ax, (slug, aoi) in zip(axes.ravel(), aois.items()):
        for label in PRODUCTS:
            sub = catalog[(catalog["slug"] == slug) & (catalog["product"] == label) & catalog["pass_all"]]
            big = label != "PlanetScope"     # make the rarer SkySat dots easier to see
            ax.scatter(sub["acquired"], sub["cloud_cover"], s=45 if big else 10,
                       marker="^" if big else "o", alpha=0.9 if big else 0.6,
                       color=colors.get(label, "gray"), label=label, zorder=3 if big else 2)
        ax.axvline(war_start, color="darkred", lw=1.5)
        ax.set_title(aoi["name"], fontsize=10)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
    axes[0, 0].legend(fontsize=8)
    for r in range(rows_n):
        axes[r, 0].set_ylabel("Cloud cover (0-1)")
    fig.suptitle("Usable scenes over time (red line = war starts)")
    plt.tight_layout()
    plt.show()

## Step 10: Save the catalog

We save three files in `OUT_DIR`:

- `catalog_all_sites.csv`: every scene found, with the `pass_*` columns, so you can refilter later without searching again.
- `catalog_usable_all_sites.csv`: only the scenes that passed the filters.
- `catalog_summary.csv`: Table 1.

CSV files open in Excel or load into pandas with `pd.read_csv(...)`.

In [ ]:
if catalog.empty:
    print("Nothing to save.")
else:
    catalog.to_csv(out_dir / "catalog_all_sites.csv", index=False)
    catalog[catalog["pass_all"]].to_csv(out_dir / "catalog_usable_all_sites.csv", index=False)
    summary.to_csv(out_dir / "catalog_summary.csv", index=False)
    for f in sorted(out_dir.glob("catalog_*.csv")):
        print("saved", f)

## Check your work

- `catalog_all_sites.csv` should have one row per scene, and `pass_all` should be `True` only where both enabled filters pass.
- Pick one scene `id` from the usable file. Every `id` is unique. You can use it later to order that scene (Lab 1, Step 7).
- Set `MAX_CLOUD = 1.0` and `MIN_COVERAGE = 0` and rerun from Step 7. Every scene should now pass. If not, something is off.

**Questions for discussion**

1. Which site has the most usable PlanetScope scenes? Which has the fewest? What might explain the difference (cloud, season, latitude, smoke or haze)?
2. Compare the months before the war to the months after. Did imagery get better, worse, or stay the same? Does the catalog tell you anything about whether Planet changed how it collected over these sites? What would you need to check to be sure?
3. SkySat is tasked, not routine. If a site has SkySat collects, what does that suggest about who asked for them, and why?
4. The cloud number covers the whole scene, not your AOI. Name one way a scene could pass the cloud filter yet still hide the tanks. Name one way a scene could fail but still be usable.
5. What are the policy risks of saying "no imagery, so nothing changed"?

**Next:** `01_search_and_retrieve_planet_imagery.ipynb` (Step 7) shows how to order clipped copies of the scenes you choose from this catalog.